# Tarefa

1. Treinar os seguintes modelos (não condicionais) utilizando o conjunto de **treinamento** MNIST (ou Fashion MNIST):

- GAN (com camadas completamente conectadas, isto é, GAN **NÃO** convolucional)
- DCGAN (convolucional)
- WGAN (convolucional)

Observação: A utilização dos laboratórios anteriores é permitida. Nesse caso, salvar os modelos treinados e carregar tais modelos aqui.

2. Comparar os modelos utilizando as curvas de MPR para diferentes valores do parâmetro `k` (mostre os gráficos com legendas para os modelos). Utilize o conjunto de **teste** MNIST (ou Fashion MNIST) para a aproximação da variedade dos dados reais.

- Qual o comportamento da precisão e revocação ao se aumentar o valor de k?

3. Varie o número de imagens `N=[1000, 2500, 5000, 7500, 10000]` utilizadas (utilize o mesmo valor para o número de imagens reais e o número de imagens geradas) no cálculo da precisão e revocação. Observe o comportamento das curvas de precisão e revocação (para `k=[3, 5, 10]`) para cada modelo.

- Mostrar 3 gráficos, um para cada modelo.

**Entregáveis**:
1. Notebook `.ipynb`.
2. Relatório `.pdf`:

    - Reporte e comente os resultados no relatório.

    - Incluir gráficos gerados.


# Classificador

## Modelo

In [ ]:
# Classificador dividido em dois nn.Sequential: feature_extractor + classifier_head

# Extrator de features
feature_extractor = nn.Sequential(
    nn.Conv2d(1, 32, 3), nn.ReLU(), nn.MaxPool2d(2),    # -> (B,32,13,13)
    nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),   # -> (B,64,5,5)
    nn.Conv2d(64, 128, 3), nn.ReLU(),                   # -> (B,128,3,3)
    nn.AdaptiveAvgPool2d(1),                            # -> (B,128,1,1)
    nn.Flatten()                                        # -> (B,128)
).to(device)

# Cabeça classificadora
classifier_head = nn.Sequential(
    nn.Linear(128, 10)                                  # -> (B,10)
).to(device)

# Modelo completo
full_model = nn.Sequential(
    feature_extractor,
    classifier_head
).to(device)

In [ ]:
# Extrator de features
summary(feature_extractor, input_size=(1, 1, 28, 28))

In [ ]:
# Cabeça classificadora
summary(classifier_head, input_size=(1, 128))

## Treinamento do classificador

In [ ]:
## Função de perda e otimizadores
loss_fn = nn.CrossEntropyLoss(reduction='sum')
optimizer = torch.optim.Adam(full_model.parameters(), lr=1e-3)

## Preparação do conjunto de dados
train_dl = DataLoader(mnist_dataset, batch_size=128, shuffle=True,  drop_last=True)
test_dl  = DataLoader(mnist_test,    batch_size=512, shuffle=False, drop_last=False)

In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()                                               # Coloca o modelo em modo de treinamento (ativa dropout, batchnorm, etc.)
    total_loss, total, correct = 0.0, 0, 0                      # Inicializa acumuladores de perda, número de exemplos e acertos
    for x, y in dataloader:                                     # Itera pelos lotes de dados
        x, y = x.to(device), y.to(device)                       # Move entradas e rótulos para o dispositivo (CPU/GPU)
        logits = model(x)                                       # Calcula as predições do modelo
        loss = loss_fn(logits, y)                               # Calcula a perda entre predições e rótulos
        optimizer.zero_grad()                                   # Zera gradientes acumulados
        loss.backward()                                         # Calcula gradientes via backpropagation
        optimizer.step()                                        # Atualiza os parâmetros do modelo
        total_loss += loss.item()                               # Acumula a perda do lote
        correct    += (logits.argmax(1) == y).sum().item()      # Conta acertos comparando previsão com rótulo
        total      += x.size(0)                                 # Conta número de exemplos processados
    return total_loss/total, correct/total                      # Retorna perda média por exemplo e acurácia

In [ ]:
@torch.no_grad()                                                 # Desativa o cálculo de gradientes (avaliação mais rápida e com menor uso de memória)
def evaluate(dataloader, model, loss_fn):
    model.eval()                                                 # Coloca o modelo em modo de avaliação (desativa dropout, batchnorm usa estatísticas fixas)
    total_loss, total, correct = 0.0, 0, 0                       # Inicializa acumuladores de perda, número de exemplos e acertos
    for x, y in dataloader:                                      # Itera pelos lotes de dados
        x, y = x.to(device), y.to(device)                        # Move entradas e rótulos para o dispositivo (CPU/GPU)
        logits = model(x)                                        # Calcula as predições do modelo
        loss = loss_fn(logits, y)                                # Calcula a perda entre predições e rótulos
        total_loss += loss.item()                                # Acumula a perda do lote
        correct    += (logits.argmax(1) == y).sum().item()       # Conta acertos comparando previsão com rótulo
        total      += x.size(0)                                  # Conta número de exemplos processados
    return total_loss/total, correct/total                       # Retorna perda média por exemplo e acurácia

In [ ]:
n_epocas = 10                                                                                       # Define o número total de épocas de treinamento
for ep in range(n_epocas):                                                                          # Loop principal sobre as épocas
    tr_loss, tr_acc = train_one_epoch(train_dl, full_model, loss_fn, optimizer)                     # Executa uma época de treinamento e retorna perda/acurácia
    te_loss, te_acc = evaluate(test_dl, full_model, loss_fn)                                        # Avalia o modelo no conjunto de teste e retorna perda/acurácia
    print(f"Época {ep + 1}/{n_epocas} | Treinamento: perda={tr_loss:.4f} acc={tr_acc:.3f} | "
          f"Teste: perda={te_loss:.4f} acc={te_acc:.3f}")                                          # Imprime resultados formatados de treino e teste


## Extração de características

In [ ]:
@torch.no_grad()                                                                                   # Desativa cálculo de gradientes (inferencia mais rápida e leve)
def get_features(dataset, feature_extractor, N: int, batch_size: int = 512) -> np.ndarray:
    feature_extractor.eval()                                                                        # Coloca o extrator de características em modo de avaliação
    N_eff = min(N, len(dataset))                                                                    # Garante que N não ultrapasse o tamanho do dataset
    subset = Subset(dataset, range(N_eff))                                                          # Seleciona os primeiros N exemplos do dataset
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, drop_last=False)              # Cria um DataLoader sem embaralhar e sem descartar amostras

    feats_list = []                                                                                 # Lista para acumular os vetores de características
    for x, _ in loader:                                                                             # Itera sobre os lotes do DataLoader
        x = x.to(device)                                                                            # Move o batch de imagens para o dispositivo (CPU/GPU)
        emb = feature_extractor(x)                                                                  # Extrai features com o modelo
        feats_list.append(emb.detach().cpu().numpy().astype(np.float32))                            # Converte para NumPy float32 e adiciona à lista
    return np.concatenate(feats_list, axis=0)                                                       # Concatena todos os lotes em um único array (N, D)

In [ ]:
@torch.no_grad()                                                                                   # Desativa cálculo de gradientes (inferência mais rápida)
def get_features_from_generator(
    gen_model,
    feature_extractor,
    N: int,
    *,
    z_dim: int = z_size,
    mode_z: str = mode_z,
    batch: int = batch_size
) -> np.ndarray:

    gen_model.eval()                                                                                # Coloca o gerador em modo de avaliação
    feature_extractor.eval()                                                                        # Coloca o extrator em modo de avaliação

    feats = []                                                                                      # Lista para acumular as features
    remaining = N                                                                                   # Número de amostras restantes a serem geradas

    while remaining > 0:                                                                            # Gera em lotes até completar N
        cur = min(batch, remaining)                                                                 # Define o tamanho do lote atual
        z = create_noise(cur, z_dim, mode_z).to(device)                                             # Gera ruído latente (cur, z_dim) e move para o dispositivo
        imgs = gen_model(z).view(cur, 1, *image_size).to(device)                                    # Gera imagens (cur, 1, 28, 28) no intervalo [-1,1]
        emb  = feature_extractor(imgs)                                                              # Extrai as features das imagens geradas
        feats.append(emb.detach().cpu().numpy().astype(np.float32))                                 # Converte para NumPy float32 e acumula
        remaining -= cur                                                                            # Atualiza o contador de amostras restantes
    return np.concatenate(feats, axis=0)                                                            # Concatena todos as features em um array (N, D)

## Precisão e Revocação de Variedades

In [ ]:
dataset = mnist_test                                                                  # Usa o conjunto de teste MNIST
feature_extractor = full_model[0]                                                     # Seleciona apenas a parte extratora de características do classificador
N = 10_000                                                                            # Número de amostras a serem processadas
batch_size = 512                                                                      # Tamanho do mini-lote

### Features reais

In [ ]:
fake_features = get_features_from_generator(                                          # Extrai as features das imagens geradas pelo gerador
    gen_model,                                                                        # Modelo gerador previamente treinado
    feature_extractor=feature_extractor,                                              # Extrator de características do classificador
    N=N,                                                                              # Número de imagens a serem geradas e processadas
    z_dim=z_size,                                                                     # Dimensão do vetor de ruído de entrada do gerador
    mode_z=mode_z,                                                                    # Distribuição do ruído (uniforme ou normal)
    batch=batch_size                                                                  # Tamanho do mini-lote para geração e extração
)
print("fake_features shape:", fake_features.shape)                                    # Mostra o shape esperado: (N, dimensão do embedding), ex.: (10000, 128)

### Features geradas

In [ ]:
fake_features = get_features_from_generator(                                          # Extrai as features das imagens geradas pelo gerador
    gen_model,                                                                        # Modelo gerador previamente treinado
    feature_extractor=feature_extractor,                                              # Extrator de características do classificador
    N=N,                                                                              # Número de imagens a serem geradas e processadas
    z_dim=z_size,                                                                     # Dimensão do vetor de ruído de entrada do gerador
    mode_z=mode_z,                                                                    # Distribuição do ruído (uniforme ou normal)
    batch=batch_size                                                                  # Tamanho do mini-lote para geração e extração
)
print("fake_features shape:", fake_features.shape)                                    # Mostra o shape esperado: (N, dimensão do embedding), ex.: (10000, 128)

### Cálculo

In [ ]:
metrics = compute_prdc(real_features, fake_features, nearest_k=5)                     # Calcula métricas de Precisão e Revocação de Variedades (PRDC)
print(f"Precisão: {metrics['precision']:.4f}")                                        # Exibe a precisão
print(f"Revocação: {metrics['recall']:.4f}")                                          # Exibe a revocação

In [ ]:
def mpr_curve_by_k(real_feats, fake_feats, ks):
    prec, rec = [], []                                               # Listas para armazenar precisão e revocação
    for k in ks:                                                     # Itera sobre os valores de k fornecidos
        print(f'Calculando MPR para k={k}...')                       # Mostra no console qual k está sendo processado
        m = compute_prdc(real_feats, fake_feats, nearest_k=k)        # Calcula precisão e revocação para o valor atual de k
        prec.append(m['precision'])                                  # Adiciona precisão calculada à lista
        rec.append(m['recall'])                                      # Adiciona revocação calculada à lista
    return np.array(rec), np.array(prec)                             # Retorna arrays de revocação e precisão

In [ ]:
def plot_mpr_curve(recall_arr, precision_arr, ks, title='Curva MPR (variando k)'):
    plt.figure()                                                              # Cria uma nova figura para o gráfico
    plt.plot(recall_arr, precision_arr, marker='o', label='GAN (FC)')         # Plota a curva de revocação vs. precisão
    for r, p, k in zip(recall_arr, precision_arr, ks):                        # Itera sobre cada ponto (r, p) com respectivo k
        plt.text(r, p, f'k={k}')                                              # Anota o valor de k próximo ao ponto correspondente
    plt.xlim(0, 1)                                                            # Define limites do eixo X entre 0 e 1
    plt.ylim(0, 1)                                                            # Define limites do eixo Y entre 0 e 1
    plt.xlabel('Revocação')                                                   # Rótulo do eixo X
    plt.ylabel('Precisão')                                                    # Rótulo do eixo Y
    plt.title(title)                                                          # Define o título do gráfico
    plt.grid(True)                                                            # Ativa a grade no gráfico
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))                    # Posiciona a legenda à esquerda fora da área do gráfico
    plt.show()                                                                # Exibe o gráfico na tela

In [ ]:
ks = list(range(1, 11))                                         # Define a lista de valores de k de 1 até 10
rec, prec = mpr_curve_by_k(real_features, fake_features, ks)    # Calcula arrays de revocação e precisão para cada k

In [ ]:
plot_mpr_curve(rec, prec, ks, title='Curva MPR (variando k)')   # Plota a curva MPR de revocação vs. precisão para diferentes valores de k

In [ ]:
def mpr_sweep_Nk(
    mnist_test_dataset,
    gen_model,
    feature_extractor,
    Ns=(1000, 2500, 5000, 7500, 10000),
    ks=(3, 5, 10),
    *,
    model_name='GAN (FC)',
    batch_size=512,
    z_dim=None,
    mode_z=None
) -> pd.DataFrame:

    linhas = []                                                                                         # Lista para armazenar resultados intermediários
    for N in Ns:                                                                                        # Itera pelos diferentes valores de N
        real_feats = get_features(mnist_test_dataset, feature_extractor, N=N, batch_size=batch_size)    # Extrai features reais dos primeiros N exemplos
        fake_feats = get_features_from_generator(                                                       # Extrai features de N imagens geradas
            gen_model,
            feature_extractor,
            N=N,
            z_dim=z_dim,
            mode_z=mode_z,
            batch=batch_size
        )
        for k in ks:                                                                             # Itera sobre os diferentes valores de k
            m = compute_prdc(                                                                    # Calcula métricas de precisão e revocação
                real_features=real_feats.astype('float32'),
                fake_features=fake_feats.astype('float32'),
                nearest_k=k
            )
            linhas.append({                                                                      # Adiciona linha com resultados no formato dict
                'modelo':   model_name,
                'N':        int(N),
                'k':        int(k),
                'precision':float(m['precision']),
                'recall':   float(m['recall'])
            })
            print(f"[modelo={model_name}] N={N} | k={k} -> Precisão={m['precision']:.4f} | Revocação={m['recall']:.4f}")  # Exibe no console os resultados do par (N, k)

    return pd.DataFrame(linhas)                                                                  # Retorna um DataFrame com todos os resultados

In [ ]:
df_gan = mpr_sweep_Nk(                                           # Executa a varredura de N e k para calcular MPR
    mnist_test_dataset=mnist_test,                               # Conjunto de teste do MNIST usado para extrair features reais
    gen_model=gen_model,                                         # Gerador treinado que produz imagens sintéticas
    feature_extractor=full_model[0],                             # Extrator de características do classificador (parte inicial do modelo)
    Ns=(1000, 2500, 5000, 7500, 10000),                          # Conjunto de valores de N a serem testados
    ks=(3, 5, 10),                                               # Conjunto de valores de k a serem testados
    model_name='GAN (FC)',                                       # Nome do modelo para identificar os resultados no DataFrame
    batch_size=512,                                              # Tamanho dos lotes para processar dados
    z_dim=z_size,                                                # Dimensão do vetor de ruído do gerador
    mode_z=mode_z)                                               # Tipo de ruído usado (normal ou uniforme)

In [ ]:
display(df_gan)   # Exibe o DataFrame df_gan em formato tabular

In [ ]:
def plot_mpr_df(df: pd.DataFrame):

    modelos = df['modelo'].unique().tolist()                                      # Lista os nomes únicos de modelos presentes no DataFrame
    n_models = len(modelos)                                                       # Conta quantos modelos distintos existem

    fig, axes = plt.subplots(                                                     # Cria figura com subplots
        n_models, 2,                                                              # n_models linhas, 2 colunas (Precisão e Revocação)
        figsize=(12, 4 * n_models),                                               # Ajusta o tamanho da figura proporcional ao número de modelos
        dpi=110,                                                                  # Define resolução em pontos por polegada
        constrained_layout=True                                                   # Ajusta automaticamente os espaçamentos
    )

    if n_models == 1:                                                             # Caso haja apenas 1 modelo
        axes = np.array([axes])                                                   # Garante que axes seja indexável como array bidimensional

    for i, modelo in enumerate(modelos):                                          # Itera sobre cada modelo
        df_model = df[df['modelo'] == modelo].copy()                              # Filtra o DataFrame apenas para esse modelo

        ax_left = axes[i, 0]                                                      # Eixo à esquerda para precisão
        sns.lineplot(                                                             # Plota curva de precisão
            data=df_model, x='N', y='precision',
            hue='k', marker='o', ax=ax_left
        )
        ax_left.set_title(f'Precisão vs N ({modelo})')                            # Título do gráfico de precisão
        ax_left.set_xlabel('N (número de imagens reais = geradas)')               # Rótulo eixo X
        ax_left.set_ylabel('Precisão')                                            # Rótulo eixo Y
        ax_left.grid(True, linestyle='--', alpha=0.4)                             # Ativa grade com estilo tracejado
        ax_left.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='k')     # Legenda deslocada para a direita

        ax_right = axes[i, 1]                                                     # Eixo à direita para revocação
        sns.lineplot(                                                             # Plota curva de revocação
            data=df_model, x='N', y='recall',
            hue='k', marker='o', ax=ax_right
        )
        ax_right.set_title(f'Revocação vs N ({modelo})')                          # Título do gráfico de revocação
        ax_right.set_xlabel('N (número de imagens reais = geradas)')              # Rótulo eixo X
        ax_right.set_ylabel('Revocação')                                          # Rótulo eixo Y
        ax_right.grid(True, linestyle='--', alpha=0.4)                            # Ativa grade com estilo tracejado
        ax_right.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='k')    # Legenda deslocada para a direita

    plt.show()                                                                    # Exibe a figura final com todos os subplots